In [ ]:
!pip install -q datasets transformers sentencepiece

In [ ]:
import os
import numpy as np
import pandas as pd

from datasets import load_dataset, DatasetDict
from transformers import BartTokenizer

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
project_dir = "/content/drive/MyDrive/nlp_summarization_project"
os.makedirs(project_dir, exist_ok=True)

print("Project directory:", project_dir)

In [ ]:
dataset = load_dataset("cnn_dailymail", "3.0.0")
dataset

In [ ]:
print(dataset)
print("\nTrain columns:", dataset["train"].column_names)

DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 11490
    })
})

Train columns: ['article', 'highlights', 'id']


In [ ]:
sample = dataset["train"][0]

print("ARTICLE:\n")
print(sample["article"][:3000])

print("\n" + "="*100 + "\n")

print("HIGHLIGHTS / SUMMARY:\n")
print(sample["highlights"])

print("\n" + "="*100 + "\n")

print("ID:\n")
print(sample["id"])

ARTICLE:

LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don't plan to be one of those people who, as soon as they turn 18, suddenly buy themselves a massive sports car collection or something similar," he told an Australian interviewer earlier this month. "I don't think I'll be particularly extravagant. "The things I like buying are things that cost about 10 pounds -- books and CDs and DVDs." At 18, Radcliffe will be able to gamble in a casino, buy a drink in a pub or see the horror film "Hostel: Part II," currently six places below his number one movie on the UK box office chart. Deta

In [ ]:
train_ds = dataset["train"]
val_ds = dataset["validation"]
test_ds = dataset["test"]

print("Train size:", len(train_ds))
print("Validation size:", len(val_ds))
print("Test size:", len(test_ds))

Train size: 287113
Validation size: 13368
Test size: 11490


In [ ]:
def count_empty_examples(ds):
    empty_article = 0
    empty_summary = 0

    for ex in ds:
        if ex["article"] is None or ex["article"].strip() == "":
            empty_article += 1
        if ex["highlights"] is None or ex["highlights"].strip() == "":
            empty_summary += 1

    return empty_article, empty_summary

train_empty_article, train_empty_summary = count_empty_examples(train_ds)
val_empty_article, val_empty_summary = count_empty_examples(val_ds)
test_empty_article, test_empty_summary = count_empty_examples(test_ds)

print("TRAIN - empty article:", train_empty_article, "| empty summary:", train_empty_summary)
print("VAL   - empty article:", val_empty_article, "| empty summary:", val_empty_summary)
print("TEST  - empty article:", test_empty_article, "| empty summary:", test_empty_summary)

TRAIN - empty article: 0 | empty summary: 0
VAL   - empty article: 0 | empty summary: 0
TEST  - empty article: 0 | empty summary: 0


In [ ]:
def check_duplicate_ids(ds):
    ids = ds["id"]
    unique_ids = set(ids)
    return len(ids), len(unique_ids), len(ids) - len(unique_ids)

train_total, train_unique, train_dupes = check_duplicate_ids(train_ds)
val_total, val_unique, val_dupes = check_duplicate_ids(val_ds)
test_total, test_unique, test_dupes = check_duplicate_ids(test_ds)

print("TRAIN -> total:", train_total, "| unique:", train_unique, "| duplicates:", train_dupes)
print("VAL   -> total:", val_total, "| unique:", val_unique, "| duplicates:", val_dupes)
print("TEST  -> total:", test_total, "| unique:", test_unique, "| duplicates:", test_dupes)

TRAIN -> total: 287113 | unique: 287113 | duplicates: 0
VAL   -> total: 13368 | unique: 13368 | duplicates: 0
TEST  -> total: 11490 | unique: 11490 | duplicates: 0


In [ ]:
def get_text_stats(ds):
    article_char_lens = []
    summary_char_lens = []
    article_word_lens = []
    summary_word_lens = []

    for ex in ds:
        article = ex["article"]
        summary = ex["highlights"]

        article_char_lens.append(len(article))
        summary_char_lens.append(len(summary))

        article_word_lens.append(len(article.split()))
        summary_word_lens.append(len(summary.split()))

    return {
        "article_char_mean": np.mean(article_char_lens),
        "article_char_max": np.max(article_char_lens),
        "summary_char_mean": np.mean(summary_char_lens),
        "summary_char_max": np.max(summary_char_lens),
        "article_word_mean": np.mean(article_word_lens),
        "article_word_max": np.max(article_word_lens),
        "summary_word_mean": np.mean(summary_word_lens),
        "summary_word_max": np.max(summary_word_lens),
    }

In [ ]:
pd.DataFrame([train_stats])

,article_char_mean,article_char_max,summary_char_mean,summary_char_max,article_word_mean,article_word_max,summary_word_mean,summary_word_max
0,4033.661722,15925,294.77039,7388,691.870326,2347,51.574101,1296


In [ ]:
model_name = "facebook/bart-base"
tokenizer = BartTokenizer.from_pretrained(model_name)

In [ ]:
def get_token_length_stats(ds, tokenizer, article_col="article", summary_col="highlights"):
    article_token_lens = []
    summary_token_lens = []

    for ex in ds:
        article_len = len(tokenizer.encode(ex[article_col], truncation=False))
        summary_len = len(tokenizer.encode(ex[summary_col], truncation=False))

        article_token_lens.append(article_len)
        summary_token_lens.append(summary_len)

    stats = {
        "article_token_mean": np.mean(article_token_lens),
        "article_token_max": np.max(article_token_lens),
        "article_token_p90": np.percentile(article_token_lens, 90),
        "article_token_p95": np.percentile(article_token_lens, 95),
        "article_token_p99": np.percentile(article_token_lens, 99),
        "summary_token_mean": np.mean(summary_token_lens),
        "summary_token_max": np.max(summary_token_lens),
        "summary_token_p90": np.percentile(summary_token_lens, 90),
        "summary_token_p95": np.percentile(summary_token_lens, 95),
        "summary_token_p99": np.percentile(summary_token_lens, 99),
    }

    return stats

In [ ]:
token_stats = get_token_length_stats(train_ds, tokenizer)
pd.DataFrame([token_stats])

,article_token_mean,article_token_max,article_token_p90,article_token_p95,article_token_p99,summary_token_mean,summary_token_max,summary_token_p90,summary_token_p95,summary_token_p99
0,870.020476,4677,1463.0,1715.0,2117.0,67.692079,2428,100.0,116.0,153.0


In [ ]:
max_input_length = 512
max_target_length = 128

print("max_input_length =", max_input_length)
print("max_target_length =", max_target_length)

max_input_length = 512
max_target_length = 128


In [ ]:
def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["article"],
        max_length=max_input_length,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        text_target=examples["highlights"],
        max_length=max_target_length,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:
tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

Map:   0%|          | 0/287113 [00:00<?, ? examples/s]

Map:   0%|          | 0/13368 [00:00<?, ? examples/s]

Map:   0%|          | 0/11490 [00:00<?, ? examples/s]

In [ ]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 11490
    })
})

In [ ]:
tokenized_dataset["train"][0].keys()

dict_keys(['input_ids', 'attention_mask', 'labels'])

In [ ]:
example = tokenized_dataset["train"][0]

print("Input ids length:", len(example["input_ids"]))
print("Attention mask length:", len(example["attention_mask"]))
print("Labels length:", len(example["labels"]))

Input ids length: 512
Attention mask length: 512
Labels length: 128


In [ ]:
decoded_input = tokenizer.decode(
    tokenized_dataset["train"][0]["input_ids"],
    skip_special_tokens=True
)

decoded_label = tokenizer.decode(
    tokenized_dataset["train"][0]["labels"],
    skip_special_tokens=True
)

print("DECODED INPUT:\n")
print(decoded_input[:2000])

print("\n" + "="*100 + "\n")

print("DECODED LABEL:\n")
print(decoded_label)

DECODED INPUT:

LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don't plan to be one of those people who, as soon as they turn 18, suddenly buy themselves a massive sports car collection or something similar," he told an Australian interviewer earlier this month. "I don't think I'll be particularly extravagant. "The things I like buying are things that cost about 10 pounds -- books and CDs and DVDs." At 18, Radcliffe will be able to gamble in a casino, buy a drink in a pub or see the horror film "Hostel: Part II," currently six places below his number one movie on the UK box office chart

In [ ]:
save_path = f"{project_dir}/tokenized_cnn_dailymail_bart_base"
tokenized_dataset.save_to_disk(save_path)

print("Saved tokenized dataset to:", save_path)

In [ ]:
config_text = f"""
model_name = {model_name}
max_input_length = {max_input_length}
max_target_length = {max_target_length}
dataset_name = abisee/cnn_dailymail
processed_dataset_path = {save_path}
"""

with open(f"{project_dir}/preprocessing_config.txt", "w") as f:
    f.write(config_text)

print("Config saved.")

Config saved.
